# GeoBrain API demo

This notebook demonstrates how to: 
1. Build and save geoJSON files from Allen atlas. 
2. Compute region-level score tables
3. Render brain regions 
4. Compare groups: subtraction, significance testing, and diverging color maps

- Dashboard usage: `docs/dashboard_usage.md`
- Understanding scores: `docs/score_definitions.md`
- API reference: `docs/api_reference.md`

This notebook demonstrates the Python API. If you prefer an interactive workflow, the same functionality is available through the GeoBrain dashboard.

In [ ]:
import geobrain

### 1. Build and save geoJSON files from Allen atlas. 

geobrain renders Allen atlas regions as GeoJSON polygons. In this step, we load the Allen Mouse Brain Atlas annotation volume and ontology, both of which are openly provided by the Allen Institute, and extract slices from a given orientation (e.g., coronal, sagittal, or horizontal) at different AP levels (in mm relative to Bregma). The annotation volume defines the region identity of every voxel in the atlas, while the ontology provides the hierarchical brain-region metadata used to label and annotate the resulting polygons.

Note that, in this example, we are not saving the geoJSOn file to disk. To save it, you need to call 

```python
geobrain.save_geojson(
    geojson_obj,
    out_path=geojson_path,
)
```

In [ ]:
volume = geobrain.load_annotation_volume(resolution_um=25)
structure_df = geobrain.load_structure_graph()
geojson_obj = geobrain.build_geojson(
	volume=volume,
	structure_df=structure_df,
	orientation="coronal",
	resolution_um=25,
	coords_mm=[-2.0],  # Bregma -2.0 mm
	min_area_px=5,
	simplify_px=0.8,
	polygon_mode="contour",
	smooth_sigma=1.0,
)

### 2. Compute region-level score tables

Score tables summarize the distribution of detected objects across brain regions. These scores can later be visualized as choropleth maps.

The available scores and normalization methods (Within, Reference, Pooled, Group) are described in `docs/score_definitions.md`.

Note that, in this example, we are nos saving the scores to disk. To save it, you need to call: 
```python
save_scores(
    data_dir = "path/to/dir",
    out_path = "path/to/dir",
    metadata_path = "path/to/dir",
    metadata_sep = ";",
    group_col= ["group", "sex"], 
)
``` 

In [ ]:
scores = geobrain.score_table(
	data_dir="/path/to/dir",
	metadata_path="/path/to/metadata",
	metadata_sep=";",
	group_col=["group", "sex"],
)

### 3. Render brain regions

Finally, combine the GeoJSON slice and score table to generate an interactive Plotly figure.

Note that, to save the images, we need to call the function `save_geojson`: 

```python
geobrain.save_figure(
    fig = fig, 
    out_dir = "/path/to/dir", 
)
``` 

In [ ]:
score_df = scores[
	scores["group_label"] == "12m_female"
]  # we will render just one group in this case

In [ ]:
geobrain.render_brain_slice(
	geojson_obj,
	score_df,
	value_col="relative_abundance_z",
)

### 4. Compare groups (subtraction, significance testing, diverging color maps)

Instead of comparing two atlas figures side by side, `geobrain.delta` computes per-region differences and renders them on a single diverging colormap centered at zero. Three statistical modes are available depending on your design:

- `test_two_sample`: compare two groups per region (t-test or Mann-Whitney)
- `test_one_sample`: test one group against a reference value (e.g. 0), when there's no second group to compare against
- `test_multi_sample`: n-way ANOVA across one or more factors (e.g. group × sex), for more than two groups

This section walks through the two-group case end to end, then shows the one-sample and multi-sample tests briefly.

In [ ]:
# Split the combined score table (from step 2) into the two groups being compared.
score_a = scores[scores["group_label"] == "12m_female"]
score_b = scores[scores["group_label"] == "12m_male"]

delta_df = geobrain.compute_delta(
	score_a,
	score_b,
	value_col="relative_abundance_z",
	label_a="Female",
	label_b="Male",
)
delta_df.head()

#### Significance testing

`test_two_sample` runs a per-region test on **per-animal** values, not the aggregated score table, so we go back one step and use per-animal object counts, split into the same two groups via metadata.

In [ ]:
raw_counts = geobrain.compute_animal_region_counts(
	geobrain.load_refatlas_regions(data_dir="/path/to/dir")
)

metadata = geobrain.MetadataConfig(
	metadata_path="/path/to/metadata",
	sep=";",
	animal_col="animal",
	group_col=["group", "sex"],
)
raw_counts, _ = metadata.merge_and_add_groups(raw_counts)

raw_a = raw_counts[raw_counts["group_label"] == "12m_female"]
raw_b = raw_counts[raw_counts["group_label"] == "12m_male"]

sig_df = geobrain.test_two_sample(raw_a, raw_b, value_col="objects")
sig_df.head()

In [ ]:
# Merge significance into the delta table and gray out non-significant regions.
delta_masked = geobrain.apply_significance_mask(delta_df, sig_df, value_col="delta")
delta_masked.head()

#### Rendering: raw delta vs. significance

Two ways to color the comparison map:
- **value** — the raw delta, grayed out (`NaN`) where not significant
- **significance** — `-log10(p_adj)`, signed by effect direction, so the colorbar communicates both direction and strength of evidence

In [ ]:
geobrain.render_brain_slice(
	geojson_obj,
	delta_masked,
	value_col="delta_masked",
	color_continuous_scale="RdBu_r",
	color_continuous_midpoint=0,  # keeps zero centered so +/- delta are comparable
	title="Δ relative abundance (Female − Male)",
)

In [ ]:
delta_masked["significance_color"] = geobrain.significance_color(delta_masked, effect_col="effect")

geobrain.render_brain_slice(
	geojson_obj,
	delta_masked,
	value_col="significance_color",
	color_continuous_scale="RdBu_r",
	color_continuous_midpoint=0,
	title="Significance: Female vs. Male",
)

#### More than two groups: one-sample and multi-sample tests

- `test_one_sample` asks whether a region's signal is significantly different from a reference value (0 by default) within a single cohort — no second group needed.
- `test_multi_sample` runs a full factorial ANOVA across one or more factors (e.g. `group`, `sex`, and their `group:sex` interaction) and returns one row per (region, term), so you choose which term to visualize.

In [ ]:
# One-sample: is relative abundance significantly different from 0 within a single group?
sig_one = geobrain.test_one_sample(raw_a, value_col="objects", popmean=0.0)
sig_one.head()

In [ ]:
# Multi-sample: full factorial ANOVA across group and sex.
sig_multi = geobrain.test_multi_sample(
	raw_counts,
	value_col="objects",
	factor_cols=["group", "sex"],
)
print(sig_multi["term"].unique())  # ['group', 'sex', 'group:sex']

# ANOVA has no single direction with 2+ groups, so significance is unsigned
# and rendered on a sequential colormap instead of a diverging one.
group_term = sig_multi[sig_multi["term"] == "group"].copy()
group_term["significance_color"] = geobrain.significance_color(group_term, effect_col=None)

geobrain.render_brain_slice(
	geojson_obj,
	group_term,
	value_col="significance_color",
	zmin=0.0,
	zmax=group_term["significance_color"].max(),
	color_continuous_scale="Viridis",
	title="ANOVA significance: group",
)